# MCP-Powered Personal Writer's Assistant

Features:

1. Analyse my previous writing style with **structured output**.
2. Draft in that style with a **Writer Agent**.
3. Hand off to a strict **Editor Agent**.
4. Integrate **MCP concepts** by creating a custom MCP server for writing preferences, British-English rules, and draft quality checks.


In [ ]:
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pathlib import Path
import textwrap

load_dotenv(override=True)


## Structured Style Analysis


In [ ]:
class StyleGuide(BaseModel):
    tone: str = Field(description="Overall tone of the writing")
    sentence_structure: str = Field(description="Sentence length, rhythm and complexity")
    vocabulary: str = Field(description="Word choice and expression style")

analyser_agent = Agent(
    name="Style Analyser",
    instructions=(
        "You analyse a writing sample and extract style characteristics. "
        "Pay attention to British English markers."
    ),
    model="gpt-4o-mini",
    output_type=StyleGuide,
)


In [ ]:
past_work = (
    "Life is entirely too short to drink bad coffee or write boring code. "
    "Let's build things that matter, and maybe have a laugh while doing it. "
    "We should embrace the colourful array of possibilities in software engineering."
)

with trace("Analyse Style"):
    style_result = await Runner.run(analyser_agent, past_work)

style_guide = style_result.final_output
print(style_guide.model_dump_json(indent=2))


## Custom MCP Server for Writing Support

This server exposes:

- `get_writer_profile(name)` tool
- `list_british_english_rules()` tool
- `quality_check_draft(draft)` tool
- `writer://profile/{name}` resource


In [ ]:
server_code = textwrap.dedent('''
import json
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("writer_support_server")

WRITER_PROFILES = {
    "tolu": {
        "voice": "warm, witty, practical",
        "audience": "learners and builders",
        "preferred_style": "short paragraphs, plain words, concrete examples",
        "must_use": "British English spelling",
    }
}

BRITISH_RULES = [
    "Use British spelling: colour, analyse, organise, emphasise, behaviour.",
    "Avoid robotic filler phrases and generic transitions.",
    "Prefer concrete examples over abstract claims.",
    "Keep writing concise but human and engaging.",
]

BANNED_FILLER = ["In conclusion", "delve", "today we will", "furthermore"]


@mcp.tool()
async def get_writer_profile(name: str) -> dict:
    """Return writing profile preferences for a writer."""
    return WRITER_PROFILES.get(name.lower(), WRITER_PROFILES["tolu"])


@mcp.tool()
async def list_british_english_rules() -> list[str]:
    """Return British English and writing quality rules."""
    return BRITISH_RULES


@mcp.tool()
async def quality_check_draft(draft: str) -> dict:
    """Run a lightweight quality check on a draft."""
    lower = draft.lower()
    filler_hits = [w for w in BANNED_FILLER if w.lower() in lower]
    us_spellings = []
    checks = {
        "color": "colour",
        "analyze": "analyse",
        "organize": "organise",
        "behavior": "behaviour",
    }
    for us in checks:
        if us in lower:
            us_spellings.append(us)

    return {
        "passes": len(filler_hits) == 0 and len(us_spellings) == 0,
        "filler_found": filler_hits,
        "us_spelling_found": us_spellings,
        "word_count": len(draft.split()),
        "suggestion": "Looks good" if len(filler_hits) == 0 and len(us_spellings) == 0 else "Revise and re-check",
    }


@mcp.resource("writer://profile/{name}")
async def read_writer_profile(name: str) -> str:
    profile = WRITER_PROFILES.get(name.lower(), WRITER_PROFILES["tolu"])
    return json.dumps(profile)


if __name__ == "__main__":
    mcp.run(transport="stdio")
''').lstrip()

# Robustly locate the toluwalemi contribution folder regardless of
# where Jupyter was launched from.
def _find_contribution_dir() -> Path:
    """Walk up from CWD until we find the toluwalemi folder."""
    target = Path("6_mcp") / "community_contributions" / "toluwalemi"
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        candidate = parent / target
        if candidate.is_dir():
            return candidate
        # Already inside the folder?
        if (parent / "mcp_writer_assistant.ipynb").exists():
            return parent
    return Path.cwd()  # final fallback

server_path = _find_contribution_dir() / "writer_support_server.py"
server_path.parent.mkdir(parents=True, exist_ok=True)
server_path.write_text(server_code, encoding="utf-8")
print(f"Wrote MCP server to: {server_path}")


## Inspect MCP Tools


In [ ]:
params = {"command": "uv", "args": ["run", str(server_path)]}

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    tools = await mcp_server.list_tools()

tools


## Writer + Editor Agents with MCP Integration

The Writer Agent must use MCP tools to gather profile + rules + quality feedback before finalising draft.


In [ ]:
editor_agent = Agent(
    name="Harsh Editor",
    instructions=(
        "You receive a draft and improve it. Remove generic AI filler, tighten clarity, "
        "and enforce British English spelling and grammar. Output only the final edited paragraph."
    ),
    model="gpt-4o-mini",
)


In [ ]:
new_topic = "How short recovery breaks improve coding quality and decision-making."

instructions = """
You are Style Drafter.

You MUST use available MCP tools before writing:
1) Call get_writer_profile(name='tolu')
2) Call list_british_english_rules()
3) Draft a short paragraph (2-3 sentences) on the given topic, matching the supplied style guide.
4) Call quality_check_draft(draft=<your draft>) and fix any issues found.
5) Return the improved draft.

Always use British English spelling.
After drafting, hand off to Harsh Editor for final polish.
"""

prompt = (
    f"Topic: {new_topic}\n"
    f"Style Guide JSON: {style_guide.model_dump_json()}\n"
    "Write in an engaging, practical tone with concrete examples."
)

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    writer_agent = Agent(
        name="Style Drafter",
        instructions=instructions,
        model="gpt-4o-mini",
        mcp_servers=[mcp_server],
        handoffs=[editor_agent],
    )

    with trace("MCP Writer Handoff"):
        final_result = await Runner.run(writer_agent, prompt)

print(final_result.final_output)


## Direct MCP Resource Read

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command="uv", args=["run", str(server_path)])

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        resource = await session.read_resource("writer://profile/tolu")
        print(resource.contents[0].text)
